In [ ]:
from astroquery.sdss import SDSS
from astropy import coordinates as coords
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import make_lupton_rgb
from tkinter import Tk, filedialog

# координати
ra = float(input("Enter RA (degrees): "))
dec = float(input("Enter DEC (degrees): "))

position = coords.SkyCoord(ra=ra, dec=dec, unit="deg", frame='icrs')

print("Querying SDSS...")

# 1. знайти об'єкти поблизу
matches = SDSS.query_region(position, radius='0.02 deg')

if matches is None:
    print("No objects found.")
    exit()

# 2. отримати FITS
images = SDSS.get_images(matches=matches, band=['g', 'r', 'i'])

if images is None:
    print("No images found.")
    exit()

# -----------------------------
# 3. Витягуємо канали
# -----------------------------
g = images[0][0].data.astype(float)
r = images[1][0].data.astype(float)
i = images[2][0].data.astype(float)

# -----------------------------
# 4. Очистка
# -----------------------------
g = np.nan_to_num(g)
r = np.nan_to_num(r)
i = np.nan_to_num(i)

# -----------------------------
# 5. Нормалізація
# -----------------------------
def normalize(img):
    return (img - np.min(img)) / (np.max(img) - np.min(img))

g = normalize(g)
r = normalize(r)
i = normalize(i)

# -----------------------------
# 6. Lupton RGB
# -----------------------------
rgb = make_lupton_rgb(i, r, g, Q=10, stretch=0.5)

# -----------------------------
# 7. Показ
# -----------------------------
plt.figure()
plt.imshow(rgb)
plt.axis('off')
plt.title(f"SDSS RGB (RA={ra}, DEC={dec})")
plt.show()

# -----------------------------
# 8. Збереження
# -----------------------------
save = input("Save image? (y/n): ")

if save.lower() == 'y':
    filename = f"sdss_{ra}_{dec}.png"
    plt.imsave(filename, rgb)
    print("Saved to:", filename)


Querying SDSS...
